In [2]:
# %pip install -U eai-sparsify

# REFERENCES
# https://github.com/EleutherAI/sparsify
# https://arxiv.org/abs/2406.04093v1
# https://github.com/decoderesearch/SAELens
# https://huggingface.co/EleutherAI/sae-pythia-70m-32k
# https://huggingface.co/datasets/Salesforce/wikitext

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sparsify import SaeConfig, Trainer, TrainConfig
from sparsify.data import chunk_and_tokenize


print("CUDA disponible :", torch.cuda.is_available())
print("Appareil :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun GPU CUDA")

MODEL = "EleutherAI/pythia-70m"

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train[:2%]",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenized = chunk_and_tokenize(dataset, tokenizer)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

cfg = TrainConfig(SaeConfig(), batch_size=4)
trainer = Trainer(cfg, tokenized, model)
trainer.fit()




CUDA disponible : False
Appareil : aucun GPU CUDA


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Training on modules: ['layers.0', 'layers.1', 'layers.2', 'layers.3', 'layers.4', 'layers.5']
Initializing SAEs with random seed(s) [0]
Learning rate: 5.00e-03
Weights & Biases not available, skipping logging.
Run `pip install -U wandb` if you want to use it.
Number of SAE parameters: 100_764_672
Number of model parameters: 70_426_624


Training:   0%|          | 0/5 [00:00<?, ?it/s]

RuntimeError: 0 active drivers ([]). There should only be one.